In [0]:
# vendors_raw = spark.read.parquet(
#     "/Volumes/workspace/default/gst-fraud&anomaly_detection/final_raw/vendors"
# )

# purchase_orders_raw = spark.read.parquet(
#     "/Volumes/workspace/default/gst-fraud&anomaly_detection/final_raw/purchase_orders"
# )

# vendor_invoices_raw = spark.read.parquet(
#     "/Volumes/workspace/default/gst-fraud&anomaly_detection/final_raw/vendor_invoices"
# )

# hsn_raw = spark.read.parquet(
#     "/Volumes/workspace/default/gst-fraud&anomaly_detection/final_raw/hsn_rate_schedule"
# )

# blacklist_raw = spark.read.parquet(
#     "/Volumes/workspace/default/gst-fraud&anomaly_detection/final_raw/cbic_blacklist"
# )

# ground_truth_raw = spark.read.parquet(
#     "/Volumes/workspace/default/gst-fraud&anomaly_detection/final_raw/ground_truth"
# )

# historical_raw = spark.read.parquet(
#     "/Volumes/workspace/default/gst-fraud&anomaly_detection/final_raw/historical_po_values"
# )




# print("vendors:", vendors_raw.count())
# print("purchase_orders:", purchase_orders_raw.count())
# print("vendor_invoices:", vendor_invoices_raw.count())
# print("hsn:", hsn_raw.count())
# print("blacklist:", blacklist_raw.count())
# print("ground_truth:", ground_truth_raw.count())
# print("historical:", historical_raw.count())

In [0]:
vendors_raw = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/vendors"
)

purchase_orders_raw = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/purchase_orders"
)

vendor_invoices_raw = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/vendor_invoices"
)

hsn_raw = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/hsn_rate_schedule"
)

blacklist_raw = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/cbic_blacklist"
)

ground_truth_raw = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/ground_truth"
)

historical_raw = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/historical_po_values"
)



print("vendors:", vendors_raw.count())
print("purchase_orders:", purchase_orders_raw.count())
print("vendor_invoices:", vendor_invoices_raw.count())
print("hsn:", hsn_raw.count())
print("blacklist:", blacklist_raw.count())
print("ground_truth:", ground_truth_raw.count())
print("historical:", historical_raw.count())

vendors: 1000
purchase_orders: 50480
vendor_invoices: 50100
hsn: 20
blacklist: 50
ground_truth: 50480
historical: 12000


In [0]:
#Date Cast

from pyspark.sql import functions as F

bronze_purchase_orders = purchase_orders_raw

bronze_purchase_orders = bronze_purchase_orders.withColumn(
    "po_date",
    F.to_date(F.col("po_date"), "dd/MM/yyyy")
)

#add Partition
bronze_purchase_orders = bronze_purchase_orders \
    .withColumn("year", F.year("po_date")) \
    .withColumn("month", F.month("po_date"))


#Add ingestion Timestamp
bronze_purchase_orders = bronze_purchase_orders.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)


#Schema Enforcement
bronze_purchase_orders = bronze_purchase_orders.filter(
    F.col("po_id").isNotNull()
).filter(
    F.col("vendor_id").isNotNull()
)


#Cheeck

bronze_purchase_orders.select(
    "po_id",
    "po_date",
    "year",
    "month",
    "ingestion_timestamp"
).show(5, False)

print(bronze_purchase_orders.count())

+---------+----------+----+-----+--------------------------+
|po_id    |po_date   |year|month|ingestion_timestamp       |
+---------+----------+----+-----+--------------------------+
|PO0000001|2023-01-05|2023|1    |2026-06-17 09:23:21.362425|
|PO0000002|2011-12-05|2011|12   |2026-06-17 09:23:21.362425|
|PO0000003|2023-08-01|2023|8    |2026-06-17 09:23:21.362425|
|PO0000004|2023-03-02|2023|3    |2026-06-17 09:23:21.362425|
|PO0000005|2023-08-23|2023|8    |2026-06-17 09:23:21.362425|
+---------+----------+----+-----+--------------------------+
only showing top 5 rows
50480


In [0]:
#Bronze vendor

bronze_vendors = vendors_raw.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)

print(bronze_vendors.count())

1000


In [0]:
#null handle

bronze_vendors = bronze_vendors.fillna(
    {
        "composition_flag": 0,
        "filing_history_q1": 0,
        "filing_history_q2": 0,
        "filing_history_q3": 0,
        "filing_history_q4": 0,
        "filing_history_q5": 0,
        "filing_history_q6": 0
    }
)

In [0]:
bronze_purchase_orders.count()

bronze_vendors.count()

bronze_purchase_orders.printSchema()

root
 |-- po_id: string (nullable = true)
 |-- po_date: date (nullable = true)
 |-- buyer_gstin: string (nullable = true)
 |-- buyer_state: string (nullable = true)
 |-- hsn_code: string (nullable = true)
 |-- product_desc: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit: string (nullable = true)
 |-- base_amount: double (nullable = true)
 |-- cgst_rate: double (nullable = true)
 |-- sgst_rate: double (nullable = true)
 |-- igst_rate: double (nullable = true)
 |-- cgst_amt: double (nullable = true)
 |-- sgst_amt: double (nullable = true)
 |-- igst_amt: double (nullable = true)
 |-- cess_amt: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- ewb_number: string (nullable = true)
 |-- ewb_generated_date: string (nullable = true)
 |-- place_of_supply: string (nullable = true)
 |-- delivery_state: string (nullable = true)
 |-- invoice_billing_name: string (nullable = true)
 |-- po_vendor_name: string (nullable = true)
 |-- id: long (nullab

In [0]:
print("vendors:", bronze_vendors.count())
print("purchase_orders:", bronze_purchase_orders.count())
print("vendor_invoices:", vendor_invoices_raw.count())
print("ground_truth:", ground_truth_raw.count())

vendors: 1000
purchase_orders: 50480
vendor_invoices: 50100
ground_truth: 50480


##SAVE


In [0]:
bronze_purchase_orders.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet(
        "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/purchase_orders"
    )

bronze_vendors.write \
    .mode("overwrite") \
    .parquet(
        "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/vendors"
    )

vendor_invoices_raw.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/vendor_invoices"
)

hsn_raw.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/hsn_rate_schedule"
)

blacklist_raw.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/cbic_blacklist"
)

ground_truth_raw.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/ground_truth"
)

historical_raw.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/historical_po_values"
)


display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/cbic_blacklist/,cbic_blacklist/,0,1781688297436
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/ground_truth/,ground_truth/,0,1781688297436
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/historical_po_values/,historical_po_values/,0,1781688297436
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/hsn_rate_schedule/,hsn_rate_schedule/,0,1781688297436
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/purchase_orders/,purchase_orders/,0,1781688297436
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/vendor_invoices/,vendor_invoices/,0,1781688297436
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/vendors/,vendors/,0,1781688297436
